# Requirements
- langchain
- Groq
- vector db - `FAISS` or `Chroma`
- FastAPI

In [1]:
import os 
from dotenv import load_dotenv

In [2]:
load_dotenv()
GROQ_API_KEY = os.getenv('GROQ_API_KEY')

In [3]:
os.environ['GROQ_API_KEY'] = GROQ_API_KEY

In [1]:
from langchain_community.document_loaders import PyPDFLoader
pdf = PyPDFLoader('../data/SDG.pdf')
docs = pdf.load()

C:\Users\imad kasse\AppData\Local\Temp\ipykernel_5248\2663389079.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader


In [ ]:
question_gen = ''
for page in docs: 
    question_gen+= page.page_content
    
question_gen    

'IN THE YEAR 2015, LEADERS FROM 193 COUNTRIES OF THE WORLD \nCAME TOGETHER TO FACE THE FUTURE.\nAnd what they saw was daunting. Famines. Drought. Wars. Plagues. Poverty. \nNot just in some faraway place, but in their own cities and towns and villages.\nThey knew things didn’t have to be this way. They knew we had enough \nfood to feed the world, but that it wasn’t getting shared. They knew there \nwere medicines for HIV and other diseases, but they cost a lot. They knew \nthat earthquakes and floods were inevitable, but that the high death \ntolls were not. \nThey also knew that billions of people worldwide shared their hope for a \nbetter future.\nSo leaders from these countries created a plan called the Sustainable \nDevelopment Goals (SDGs). This set of 17 goals imagines a future just 15 years \noff that would be rid of poverty and hunger, and safe from the worst effects of \nclimate change. It’s an ambitious plan. \nBut there’s ample evidence that we can succeed. In the past 15 yea

1. **CharacterTextSplitter:** Splits text using a single separator (e.g., `\n\n`); use for simple, structured text; suitable when splitting rules are predictable.

2. **RecursiveCharacterTextSplitter:** Recursively tries multiple separators (`\n\n`, `\n`, spaces, characters); use for PDFs and general documents; **recommended default for RAG**.

3. **TokenTextSplitter:** Splits text based on token counts using a tokenizer; use when controlling LLM token limits; recommended for strict token-based chunk sizing.

4. **Chunk Overlap:** Repeats a portion of text between consecutive chunks; use to preserve context across boundaries; recommended starting point: 50 overlap for 500-sized chunks.

5. **Best Practice:** Use `RecursiveCharacterTextSplitter.from_tiktoken_encoder()` to combine natural text splitting with token-based size measurement; recommended for token-aware RAG pipelines.

In [7]:
from langchain_classic.text_splitter import TokenTextSplitter

In [8]:
splitter_text_gen = TokenTextSplitter(chunk_size=10000, chunk_overlap=200)
chunks_question_gen = splitter_text_gen.split_text(question_gen)
chunks_question_gen

['IN THE YEAR 2015, LEADERS FROM 193 COUNTRIES OF THE WORLD \nCAME TOGETHER TO FACE THE FUTURE.\nAnd what they saw was daunting. Famines. Drought. Wars. Plagues. Poverty. \nNot just in some faraway place, but in their own cities and towns and villages.\nThey knew things didn’t have to be this way. They knew we had enough \nfood to feed the world, but that it wasn’t getting shared. They knew there \nwere medicines for HIV and other diseases, but they cost a lot. They knew \nthat earthquakes and floods were inevitable, but that the high death \ntolls were not. \nThey also knew that billions of people worldwide shared their hope for a \nbetter future.\nSo leaders from these countries created a plan called the Sustainable \nDevelopment Goals (SDGs). This set of 17 goals imagines a future just 15 years \noff that would be rid of poverty and hunger, and safe from the worst effects of \nclimate change. It’s an ambitious plan. \nBut there’s ample evidence that we can succeed. In the past 15 ye

## Convert str to documents

In [9]:
from langchain_community.docstore.document import Document
docs_question_gen =[Document(page_content= chunk) for chunk in chunks_question_gen]
docs_question_gen

[Document(metadata={}, page_content='IN THE YEAR 2015, LEADERS FROM 193 COUNTRIES OF THE WORLD \nCAME TOGETHER TO FACE THE FUTURE.\nAnd what they saw was daunting. Famines. Drought. Wars. Plagues. Poverty. \nNot just in some faraway place, but in their own cities and towns and villages.\nThey knew things didn’t have to be this way. They knew we had enough \nfood to feed the world, but that it wasn’t getting shared. They knew there \nwere medicines for HIV and other diseases, but they cost a lot. They knew \nthat earthquakes and floods were inevitable, but that the high death \ntolls were not. \nThey also knew that billions of people worldwide shared their hope for a \nbetter future.\nSo leaders from these countries created a plan called the Sustainable \nDevelopment Goals (SDGs). This set of 17 goals imagines a future just 15 years \noff that would be rid of poverty and hunger, and safe from the worst effects of \nclimate change. It’s an ambitious plan. \nBut there’s ample evidence tha

## Note : to apply text splitter should the type of element is langchain doc 

In [253]:
splitter_text_gen_new = TokenTextSplitter(chunk_size=1000, chunk_overlap=100)

In [254]:
docs_answer_gen = splitter_text_gen_new.split_documents(docs_question_gen)

In [255]:
docs_answer_gen

[Document(metadata={}, page_content='IN THE YEAR 2015, LEADERS FROM 193 COUNTRIES OF THE WORLD \nCAME TOGETHER TO FACE THE FUTURE.\nAnd what they saw was daunting. Famines. Drought. Wars. Plagues. Poverty. \nNot just in some faraway place, but in their own cities and towns and villages.\nThey knew things didn’t have to be this way. They knew we had enough \nfood to feed the world, but that it wasn’t getting shared. They knew there \nwere medicines for HIV and other diseases, but they cost a lot. They knew \nthat earthquakes and floods were inevitable, but that the high death \ntolls were not. \nThey also knew that billions of people worldwide shared their hope for a \nbetter future.\nSo leaders from these countries created a plan called the Sustainable \nDevelopment Goals (SDGs). This set of 17 goals imagines a future just 15 years \noff that would be rid of poverty and hunger, and safe from the worst effects of \nclimate change. It’s an ambitious plan. \nBut there’s ample evidence tha

## LLM

In [ ]:
from langchain_groq.chat_models import ChatGroq
llm = ChatGroq(
    model="openai/gpt-oss-20b",
    api_key=os.getenv('GROQ_API_KEY'),
    temperature=0.3
)

In [12]:
llm.invoke('what is the capital of Algeria?').content

'The capital of Algeria is **Algiers**.'

In [306]:
# Prompt
prompt_template = """
You are an expert exam-question generator.
Your task is to generate comprehensive and accurate exam questions based ONLY on the provided documentation.
DOCUMENTATION:
------------
{text}
-----------
INSTRUCTIONS:
- Generate questions covering all important concepts, facts, details, and information in the documentation.
- Do not generate answers, explanations, hints, introductions, conclusions, or summaries.
- Do not include question categories, section titles, headings, or labels.
- Do not include multiple-choice options (A, B, C, D).
- Do not include numbering, bullets, or Markdown formatting.
- Do not generate duplicate or redundant questions.
- Write all questions in English.
QUESTIONS:
"""

In [296]:
from langchain_core.prompts import PromptTemplate

prompt_ques = PromptTemplate(template=prompt_template , input_variables=['text'])

In [301]:
refine_template = ("""
You are an expert exam-question refinement assistant.
Your task is to review and improve an existing list of exam questions using the newly provided documentation.
EXISTING QUESTIONS:
------------
{existing_answer}
------------
NEW DOCUMENTATION:
------------
{text}
------------
QUESTIONS:
"""
)

In [297]:
refine_prompt_ques = PromptTemplate(template=refine_template , input_variables=['existing_answer' , 'text'])

In [298]:
from langchain_classic.chains import load_summarize_chain

In [307]:
ques_gen_chain = load_summarize_chain(llm = llm, 
                                          chain_type = "refine", 
                                          verbose = True, 
                                          question_prompt=prompt_ques, 
                                        #   refine_prompt=refine_prompt_ques
                                          )

In [308]:
question = ques_gen_chain.invoke(docs_question_gen)
question['output_text']



> Entering new RefineDocumentsChain chain...


> Entering new LLMChain chain...
Prompt after formatting:

You are an expert exam-question generator.
Your task is to generate comprehensive and accurate exam questions based ONLY on the provided documentation.
DOCUMENTATION:
------------
IN THE YEAR 2015, LEADERS FROM 193 COUNTRIES OF THE WORLD 
CAME TOGETHER TO FACE THE FUTURE.
And what they saw was daunting. Famines. Drought. Wars. Plagues. Poverty. 
Not just in some faraway place, but in their own cities and towns and villages.
They knew things didn’t have to be this way. They knew we had enough 
food to feed the world, but that it wasn’t getting shared. They knew there 
were medicines for HIV and other diseases, but they cost a lot. They knew 
that earthquakes and floods were inevitable, but that the high death 
tolls were not. 
They also knew that billions of people worldwide shared their hope for a 
better future.
So leaders from these countries created a plan called the Sustainab

RateLimitError: Error code: 429 - {'error': {'message': 'Rate limit reached for model `openai/gpt-oss-20b` in organization `org_01m6nf2bscewkr4vfd2ndjqzbx` service tier `on_demand` on tokens per day (TPD): Limit 200000, Used 199462, Requested 3992. Please try again in 24m52.128s. Need more tokens? Upgrade to Dev Tier today at https://console.groq.com/settings/billing', 'type': 'tokens', 'code': 'rate_limit_exceeded'}}

In [ ]:
# embedding model from hggingface
from langchain_huggingface import HuggingFaceEmbeddings

embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 4681.60it/s]


In [ ]:
# FAISS 
from langchain_community.vectorstores import FAISS
vector_store = FAISS.from_documents(docs_answer_gen , embedding=embeddings)

In [ ]:

question['output_text']

'Here is a comprehensive exam based on the provided text regarding the Sustainable Development Goals (SDGs). The questions are categorized by type to test different levels of comprehension, including factual recall, data interpretation, and conceptual understanding.\n\n### **Part I: Multiple Choice Questions**\n\n**1. In what year did leaders from 193 countries come together to create the Sustainable Development Goals (SDGs)?**\nA) 2000\nB) 2010\nC) 2015\nD) 2030\n\n**2. How many Sustainable Development Goals (SDGs) are there in total?**\nA) 15\nB) 17\nC) 19\nD) 193\n\n**3. By what year is the international community aiming to fulfill the SDGs?**\nA) 2025\nB) 2030\nC) 2050\nD) 2015\n\n**4. According to the text, what is the primary role of the United Nations Development Programme (UNDP) in relation to the SDGs?**\nA) To provide all medical care for HIV and other diseases.\nB) To act as a leading organization working to fulfill the SDGs and champion them so people know how to contribute

In [ ]:
from langchain_classic.chains import create_retrieval_chain
from langchain_classic.chains.combine_documents import (
    create_stuff_documents_chain
)
from langchain_classic.prompts import ChatPromptTemplate

In [ ]:
vector_store.index.ntotal
vector_store.docstore._dict

{'3db3dd02-dc3c-47a2-a496-b6bf11944960': Document(id='3db3dd02-dc3c-47a2-a496-b6bf11944960', metadata={}, page_content='IN THE YEAR 2015, LEADERS FROM 193 COUNTRIES OF THE WORLD \nCAME TOGETHER TO FACE THE FUTURE.\nAnd what they saw was daunting. Famines. Drought. Wars. Plagues. Poverty. \nNot just in some faraway place, but in their own cities and towns and villages.\nThey knew things didn’t have to be this way. They knew we had enough \nfood to feed the world, but that it wasn’t getting shared. They knew there \nwere medicines for HIV and other diseases, but they cost a lot. They knew \nthat earthquakes and floods were inevitable, but that the high death \ntolls were not. \nThey also knew that billions of people worldwide shared their hope for a \nbetter future.\nSo leaders from these countries created a plan called the Sustainable \nDevelopment Goals (SDGs). This set of 17 goals imagines a future just 15 years \noff that would be rid of poverty and hunger, and safe from the worst ef

In [ ]:
prompt = ChatPromptTemplate.from_messages([
    (
        "system",
        """You are a question-answering assistant.

        Answer the question based only on the provided context.

        Rules:
        - Do not use external knowledge.
        - Do not make up information.
        - If the context does not contain the answer,
          respond with: "I don't know based on the provided context."

        Context:
        {context}
        """
    ),
    ("human", "{input}")
])

question_answer_chain = create_stuff_documents_chain(
    llm,
    prompt
)

# 4. Create retrieval chain
retrieval_chain = create_retrieval_chain(
    vector_store.as_retriever(),
    question_answer_chain
)


In [ ]:
from pathlib import Path

# Create output file
output_file = Path("answers.txt")

with output_file.open("w", encoding="utf-8") as f:

    for index, question in enumerate(question, start=1):

        print(f"Processing question {index}/{len(question)}")

        try:
            response = retrieval_chain.invoke({
                "input": question
            })

            answer = response["answer"]

            # Display
            print(f"Question: {question}")
            print(f"Answer: {answer}")
            print("-" * 60)

            # Save
            f.write(f"Question {index}: {question}\n")
            f.write(f"Answer: {answer}\n")
            f.write("-" * 60 + "\n\n")

            f.flush()

        except Exception as e:
            print(f"Error processing question: {question}")
            print(f"Error: {e}")

print(f"All answers saved to: {output_file.resolve()}")

Processing question 1/15
Question: input_documents
Answer: I don't know based on the provided context.
------------------------------------------------------------
Processing question 2/11
Question: output_text
Answer: I don't know based on the provided context.
------------------------------------------------------------
All answers saved to: D:\learn_techs\Generative_AI_Project\interview_question\research\answers.txt
